# 03 - Dropout ablation (dropout ON vs dropout OFF)

**What this notebook does**: trains the same MiniConvNet twice on the `faithful` split - once with
`dropout_rate = 0.5`, once with `dropout_rate = 0.0` - for the **full epoch budget**, and writes a
clean 2-row table to `outputs/ablation_dropout.csv`.

**What must already exist**: split CSVs from notebook 00. Run notebook 02 first if you want the
non-ablation baselines for context.

**Design decisions baked in**
* Full epoch budget for both arms (`EPOCHS_ABLATION`) - a truncated run makes dropout look worse
  than it is, because regularised models converge more slowly.
* Both arms are collapse-checked (LESSON 3) **including the partial-collapse check** (LESSON 11).
  A collapsed or partially collapsed arm invalidates the comparison; it must be re-run, not reported.
* Raw per-sample predictions for both arms go to `outputs/predictions/` (LESSON 11).
* Everything else (seed, learning rate, clipnorm, split, activation) is held identical, so the only
  difference between the two rows is dropout.
* The ablation runs on the `flatten` head by default because that is the paper-sized variant; the
  `ARCH_FOR_ABLATION` switch below lets you repeat it for `gap`.

**Why this notebook is the one to re-run first**: its previous results (26.0% and 28.9% accuracy)
sit in the same suspiciously low range as the runs later confirmed to be partially collapsed, and
they were produced with the same setup. The old `status = ok` on both rows was the detector missing
the failure, not evidence the arms were sound.

**What "looks right"**: two rows in the ablation table, both `status = ok`, both predicting all four
classes. A modest difference in test accuracy either way is a normal outcome on a dataset this small
- what would *not* be normal is one arm sitting at ~0.25 accuracy (chance), being flagged as
collapsed, or having an all-zero column in its confusion matrix.

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

from src.config import *
from src.data_utils import resolve_data_root

ensure_dirs()
print('data root:', resolve_data_root())
print('ablation epoch budget:', EPOCHS_ABLATION)

In [ ]:
import numpy as np
import pandas as pd
import tensorflow as tf

from src.data_utils import load_split, make_split_datasets, split_counts
from src.models import build_miniconvnet, count_params
from src.train_utils import (set_global_seeds, gpu_report, compile_model, optimizer_summary,
                            class_weights_for, make_callbacks, save_history, plot_history,
                            final_epoch_summary, run_name_for)
from src.evaluate_utils import (predict, compute_metrics, detect_collapse, print_collapse_report,
                               plot_confusion_matrix, record_ablation, load_results,
                               save_predictions, confusion)

set_global_seeds(SEED)
print(gpu_report())
print('backbone activation:', MINICONVNET_ACTIVATION)

In [ ]:
# Ablation configuration - change these two lines to repeat the ablation elsewhere.
ARCH_FOR_ABLATION = 'flatten'   # 'flatten' (paper-sized) or 'gap'
SPLIT_FOR_ABLATION = 'faithful' # keep 'faithful' so the comparison is paper-comparable

print('arch :', ARCH_FOR_ABLATION)
print('split:', SPLIT_FOR_ABLATION)

## 1. Data

**Looks right**: the same counts you saw in notebook 00 for this split.

In [ ]:
sdf = load_split(SPLIT_FOR_ABLATION)
train_ds, val_ds, test_ds, frames = make_split_datasets(sdf)
class_weight = class_weights_for(SPLIT_FOR_ABLATION, frames['train']['label'].values)

print(split_counts(sdf))
print('class_weight:', class_weight if class_weight else 'None')

## 2. Ablation arm runner

One function, used twice with the only difference being `dropout_rate`. The seed is re-set inside it
so both arms start from the same initialisation stream.

In [ ]:
def run_ablation_arm(dropout_rate, tag):
    set_global_seeds(SEED)
    run_name = run_name_for('ablation', ARCH_FOR_ABLATION, SPLIT_FOR_ABLATION, tag)

    model = build_miniconvnet(ARCH_FOR_ABLATION, dropout_rate=dropout_rate)
    compile_model(model)
    n_dropout = sum(1 for l in model.layers if l.__class__.__name__ == 'Dropout')
    print('run         :', run_name)
    print('dropout_rate:', dropout_rate, '| Dropout layers in model:', n_dropout)
    print('activation  :', MINICONVNET_ACTIVATION)
    print('params      :', count_params(model))
    print('optimizer   :', optimizer_summary(model))

    history = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_ABLATION,
                        class_weight=class_weight,
                        callbacks=make_callbacks(run_name), verbose=2)
    save_history(history, run_name)
    print('\n', final_epoch_summary(history))

    y_true, y_pred, y_prob = predict(model, test_ds)
    metrics = compute_metrics(y_true, y_pred, y_prob)

    # LESSON 11: raw predictions first, so this arm can be re-checked later
    # without re-training it.
    save_predictions(run_name, y_true, y_pred, y_prob,
                     meta={'arch_variant': ARCH_FOR_ABLATION,
                           'split_variant': SPLIT_FOR_ABLATION,
                           'dropout_rate': dropout_rate,
                           'activation': MINICONVNET_ACTIVATION})

    # Also catches partial collapse (LESSON 11) - both of this ablation's arms
    # previously passed the collapse check at ~26-29% accuracy while never
    # predicting some classes at all.
    collapse = detect_collapse(history=history, kappa=metrics['cohen_kappa'],
                               mcc=metrics['mcc'], y_pred=y_pred)
    print('\ntest metrics:', {k: round(v, 4) for k, v in metrics.items()})
    print('\nconfusion matrix (an all-zero COLUMN = a class the model never predicts):')
    print(confusion(y_true, y_pred))
    print()
    print_collapse_report(collapse, run_name)

    plot_history(history, run_name)
    plot_confusion_matrix(y_true, y_pred, run_name)

    record_ablation({
        'run_name': run_name,
        'arch_variant': ARCH_FOR_ABLATION,
        'split_variant': SPLIT_FOR_ABLATION,
        'dropout_enabled': bool(dropout_rate and dropout_rate > 0),
        'dropout_rate': dropout_rate,
        'accuracy': round(metrics['accuracy'], 6),
        'f1_macro': round(metrics['f1_macro'], 6),
        'cohen_kappa': round(metrics['cohen_kappa'], 6),
        'mcc': round(metrics['mcc'], 6),
        'epochs_trained': final_epoch_summary(history)['epochs_trained'],
        'status': collapse['status'],
        'notes': (f'full budget {EPOCHS_ABLATION} epochs, identical seed/lr/clipnorm across arms; '
                  f'activation={MINICONVNET_ACTIVATION}'),
    })
    return {'run_name': run_name, 'metrics': metrics, 'collapse': collapse,
            'history': history, 'y_true': y_true, 'y_pred': y_pred}

## 3. Arm 1 - dropout ON (rate 0.5)

**Looks right**: `Dropout layers in model: 1`, and a train/val accuracy gap that is *smaller* than
the dropout-off arm.

In [ ]:
arm_on = run_ablation_arm(DROPOUT_RATE, 'dropout_on')

## 4. Arm 2 - dropout OFF (rate 0.0)

**Looks right**: `Dropout layers in model: 0` - if this prints 1, the rate was not threaded through
and the ablation is meaningless.

In [ ]:
arm_off = run_ablation_arm(0.0, 'dropout_off')

## 5. Comparison

**Looks right**: exactly 2 rows in `ablation_dropout.csv`, both `ok`, and `n_classes_predicted = 4`
for both arms. Report the delta together with the train/val gap - dropout's effect on *overfitting*
is often clearer than its effect on test accuracy at this dataset size.

**If an arm comes back `INVALID_partial_collapse`** (LESSON 11): the arm only ever chose between a
subset of the classes, so the ablation compares two broken models and the delta means nothing. The
documented fallback is `MINICONVNET_ACTIVATION = 'leaky_relu'` in `src/config.py` — see the cell
after the comparison.

In [ ]:
def gap_of(res):
    s = final_epoch_summary(res['history'])
    return round(s.get('final_accuracy', float('nan')) - s.get('final_val_accuracy', float('nan')), 4)

compare = pd.DataFrame([
    {'arm': 'dropout ON (0.5)', **{k: round(v, 4) for k, v in arm_on['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_on), 'status': arm_on['collapse']['status'],
     'n_classes_predicted': arm_on['collapse']['details'].get('n_predicted_classes')},
    {'arm': 'dropout OFF (0.0)', **{k: round(v, 4) for k, v in arm_off['metrics'].items()},
     'train_minus_val_acc': gap_of(arm_off), 'status': arm_off['collapse']['status'],
     'n_classes_predicted': arm_off['collapse']['details'].get('n_predicted_classes')},
])
print(compare[['arm', 'accuracy', 'f1_macro', 'cohen_kappa', 'mcc',
               'train_minus_val_acc', 'n_classes_predicted', 'status']].to_string(index=False))

delta = arm_on['metrics']['accuracy'] - arm_off['metrics']['accuracy']
print(f'\ntest accuracy delta (ON - OFF): {delta:+.4f}')

partial = [a for a in (arm_on, arm_off) if a['collapse'].get('partial_collapse')]
full = [a for a in (arm_on, arm_off) if a['collapse'].get('fully_collapsed')]
if full:
    print('!!! at least one arm fully collapsed - the comparison is INVALID, re-run before reporting.')
if partial:
    print(f'!!! {len(partial)} arm(s) PARTIALLY collapsed (never predict every class) - the '
          'comparison is INVALID. Do not quote the delta above.')
    print('    Fallback per LESSON 11: set MINICONVNET_ACTIVATION = "leaky_relu" in src/config.py, '
          'restart the kernel, and re-run this notebook once. Log the re-run with a config_note '
          'saying which activation it used.')
if not full and not partial:
    print('Both arms predict all four classes and passed the collapse check - the delta is reportable.')

## 6. Fallback if an arm partially collapsed — LeakyReLU (LESSON 11)

`Adam(1e-4, clipnorm=1.0)` fixed the *total* collapse of the earlier attempt, but it does not
guarantee against the milder one: a ReLU unit whose pre-activation is negative for every input has
exactly zero gradient and never comes back, and enough dead units in the narrow `Dense(16)` layers
makes whole classes unreachable. `LeakyReLU(alpha=0.1)` leaves a small gradient on the negative side,
so such a unit can recover.

The switch lives in `src/config.py` (`MINICONVNET_ACTIVATION`), applies to both the backbone and the
dense head, and **adds no parameters** — the parameter budget the paper comparison rests on is
unchanged.

**Order of operations**: only reach for this if the run above came back `INVALID_partial_collapse`.
Change the config value, restart the kernel (the config is read at import time), re-run this
notebook once, and record the activation in the `notes` column — which the arm runner already does.
Do not run both activations and then pick the better-looking one.

In [ ]:
# Read-only helper: tells you whether the fallback is needed and confirms the
# LeakyReLU model has the same parameter count. It does NOT train anything.
needs_fallback = any(a['collapse'].get('partial_collapse') or a['collapse'].get('fully_collapsed')
                     for a in (arm_on, arm_off))

print('current activation :', MINICONVNET_ACTIVATION)
print('fallback needed    :', needs_fallback)

if needs_fallback and MINICONVNET_ACTIVATION == 'relu':
    relu_params = count_params(build_miniconvnet(ARCH_FOR_ABLATION))['total_params']
    leaky_params = count_params(build_miniconvnet(ARCH_FOR_ABLATION,
                                                  activation='leaky_relu'))['total_params']
    tf.keras.backend.clear_session()
    print(f'params relu={relu_params:,} leaky_relu={leaky_params:,} '
          f'-> {"unchanged, budget preserved" if relu_params == leaky_params else "CHANGED - do not use"}')
    print('\nNext step: set MINICONVNET_ACTIVATION = "leaky_relu" in src/config.py, restart the '
          'kernel, and re-run this notebook from the top (once).')
elif needs_fallback:
    print('\nAlready on the fallback activation and still collapsing - do not keep swapping '
          'activations. Report the run as INVALID_partial_collapse and investigate the input '
          'pipeline (notebook 01) or the learning rate instead.')
else:
    print('\nNothing to do - both arms predicted all four classes.')

In [ ]:
abl = load_results('ablation')
print('outputs/ablation_dropout.csv')
print(abl.to_string(index=False))
print('\nnext: 04_cross_validation.ipynb')